# Deep Agents (Orchestrator + SubAgents with Memory, RAG, and Pydantic)

In [3]:
# Imports
import os, subprocess, tempfile
from typing import Literal, List, Optional
from pydantic import BaseModel, Field
from tavily import TavilyClient
from duckduckgo_search import DDGS
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from dotenv import load_dotenv

# LangGraph State & Memory
from langgraph.checkpoint.memory import MemorySaver

# RAG Integrations
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

load_dotenv(dotenv_path=r"..\config\.env")

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

tavily_key = os.getenv("TAVILY_API_KEY")
tavily_client = TavilyClient(api_key=tavily_key) if tavily_key else None

GOOGLE_MODEL = os.getenv("GOOGLE_MODEL")
GROQ_MODEL = os.getenv("GROQ_MODEL")

## Tools & Structured Models (Pydantic + RAG + Search: DuckDuckGo Fallback)

In [ ]:
# ---------- Pydantic Schemas ----------
class SearchResultItem(BaseModel):
    title: str = Field(description="Title of the document or webpage")
    snippet: str = Field(description="Key content summary extracted")
    source: str = Field(description="URL or source filename")

class ResearchReportSchema(BaseModel):
    summary: str = Field(description="High level overview of findings")
    results: List[SearchResultItem] = Field(description="Structured list of evidence")


# ---------- Local ChromaDB Vector Store RAG Setup ----------
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = Chroma(
    collection_name="local_docs",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

def query_local_vectorstore(query: str, k: int = 3) -> str:
    """Search the local ChromaDB vector store for indexed internal documents."""
    results = vector_store.similarity_search(query, k=k)
    if not results:
        return "No relevant local documents found."
    return "\n---\n".join([f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}" for doc in results])


# ---------- Web Search Tool ----------
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Search the web for current information, using Tavily first and falling back to DuckDuckGo."""
    if tavily_client:
        try:
            return tavily_client.search(
                query, max_results=max_results,
                include_raw_content=include_raw_content, topic=topic,
            )
        except Exception as e:
            print(f"Tavily search failed ({e}), falling back to DuckDuckGo...")
    
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
            return results if results else "No search results found."
    except Exception as ddg_err:
        return f"Both search providers failed. Error: {ddg_err}"

def run_python_code(code: str) -> str:
    """Execute a Python snippet in an isolated subprocess and return stdout/stderr."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(code)
        path = f.name
    try:
        result = subprocess.run(["python", path], capture_output=True, text=True, timeout=30)
        out = result.stdout
        if result.stderr:
            out += f"\n--- STDERR ---\n{result.stderr}"
        return out or "(no output)"
    finally:
        os.remove(path)

c:\Users\lovep\miniconda3\envs\langGraph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Local Models Setup (HuggingFace)

In [ ]:
# pip install torch transformers langchain-huggingface accelerate duckduckgo_search langchain-chroma chromadb
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# ROCm note: pass device as an integer (0), not "cuda:0" or device_map="auto" —
# those two forms are unreliable at fitting the model into VRAM on ROCm builds of torch.
LOCAL_DEVICE = 0 if torch.cuda.is_available() else -1

def load_local_causal_model(model_id: str, max_new_tokens: int = 512) -> ChatHuggingFace:
    """Load a text-only causal LM (Qwen, Llama, Mistral, Phi, DeepSeek, ...) as a chat model."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
    )

    text_gen_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        device=LOCAL_DEVICE,
    )
    llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
    return ChatHuggingFace(llm=llm)


In [ ]:
from transformers import AutoModelForMultimodalLM, AutoProcessor

def load_local_multimodal_model(model_id: str, max_new_tokens: int = 512) -> ChatHuggingFace:
    """Load a unified multimodal model (e.g. Gemma 4) as a chat model."""
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForMultimodalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
    )

    text_gen_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=processor.tokenizer,
        max_new_tokens=max_new_tokens,
        device=LOCAL_DEVICE,
    )
    llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
    return ChatHuggingFace(llm=llm)


## Models Initialization (API + Local)

In [ ]:
# ---------- Models Initialization ----------
orchestrator_model = init_chat_model(GOOGLE_MODEL)
research_model     = init_chat_model(GROQ_MODEL)
coding_model       = init_chat_model(GROQ_MODEL)

# Load local model for writing task to bypass API limitations
LOCAL_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # text-only example
# LOCAL_MODEL_ID = "google/gemma-4-it"        # multimodal example (use load_local_multimodal_model instead)

writing_model = load_local_causal_model(LOCAL_MODEL_ID)

## Sub-Agents & Orchestrator with Memory Saver Checkpointer

In [ ]:
# ---------- Subagents ----------
research_agent = {
    "name": "research-agent",
    "description": "Finds and summarizes information using both internet search and internal vector database RAG search.",
    "prompt": (
        "You are a research specialist. Use query_local_vectorstore first to check internal docs, "
        "and internet_search for external information. Always output clear, structured findings."
    ),
    "tools": [internet_search, query_local_vectorstore],
    "model": research_model,
}

coding_agent = {
    "name": "coding-agent",
    "description": "Writes, tests, and debugs code. Delegate any implementation, scripting, or debugging task here.",
    "prompt": (
        "You are a senior software engineer. Write correct, well-commented code. "
        "Use run_python_code to test snippets before returning them, and flag any "
        "errors you couldn't resolve."
    ),
    "tools": [run_python_code],
    "model": coding_model,
}

writing_agent = {
    "name": "writing-agent",
    "description": "Drafts and polishes prose: reports, summaries, docs, emails. Delegate any 'write this up' task here.",
    "prompt": (
        "You are a professional technical writer. Turn raw notes, research, or "
        "code explanations into clear, well-structured prose for the requested "
        "audience and format."
    ),
    "tools": [],
    "model": writing_model,
}

In [ ]:
# ---------- Orchestrator with Checkpointer ----------
checkpointer = MemorySaver()

orchestrator_instructions = """\
You are the orchestrator of a small team of specialist agents:
- research-agent: web and local vector search (RAG)
- coding-agent: writing/testing/debugging code
- writing-agent: drafting and polishing prose

Break the user's request into subtasks and delegate each to the right
specialist. Do not do research, coding, or long-form writing yourself.
Use stateful tracking to manage conversation history effectively.
"""

orchestrator = create_deep_agent(
    model=orchestrator_model,
    system_prompt=orchestrator_instructions,
    subagents=[research_agent, coding_agent, writing_agent],
    checkpointer=checkpointer
)

orchestrator

In [ ]:
# Running multi-turn chat with thread context
config = {"configurable": {"thread_id": "session-1"}}

result = orchestrator.invoke(
    {"messages": [{"role": "user", "content": "Research what deepagents is, write a small demo script for it, and summarize both in a short report."}]},
    config=config
)
print(result["messages"][-1].content)